# One initial descent direction with triggered resampling versus Horváth L-infinity

Comparison on the same original 50 AR(3) coefficient-recovery paths, with 2000 observations and known order 3.

Your method uses **one initial random direction** and draws another only if the best-width estimate fails the stationarity check after 200 evaluated iterations. Each attempt uses **max_it=2000**, **improvement patience=200**, the original geometric step schedule, and width-improvement tolerance 1e-10. Return the first attempt that survives and finishes; do not run more starts afterwards or apply a final stationarity filter. The unused 100-attempt guard is a safety limit, not a planned allocation.

Horváth uses the existing **no-intercept** L-infinity LP with convex-hull constraint reduction and full-data residual/duality checks. It does not fit a residual centre. Sources: [implementation](ar_tools/classical_selectors.py), [Horváth (1989), publication record](https://dml.cz/handle/10338.dmlcz/104367), [SciPy LP interface](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html).

Fit times include preparation and all triggered restarts. Both implementations are warmed up, then timed five times per process with alternating order; the median per process is recorded. File I/O, truth-based scoring and plotting are excluded. Timing repetitions repeat the same fit, rather than adding new random-start experiments.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents)
                  if (p/'Code/Code_AR_processes/ar_tools/descent.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
from output.analysis.ar3_single_start_restart_vs_horvath.run_experiment import main as run_experiment
from output.analysis.ar3_single_start_restart_vs_horvath.verify_and_report import plot_and_report
DATA_DIR=PROJECT_ROOT/'Code/Code_AR_processes/outputs/csv/ar3_single_start_restart_vs_horvath'


In [2]:
summary=run_experiment()


Completed 1/50 (0.0s)
Completed 5/50 (0.0s)
Completed 10/50 (0.0s)
Completed 15/50 (0.0s)
Completed 20/50 (0.0s)
Completed 25/50 (0.0s)
Completed 30/50 (0.0s)
Completed 35/50 (0.0s)
Completed 40/50 (0.0s)
Completed 45/50 (0.0s)
Completed 50/50 (0.0s)
                             method  processes  stationary_estimates  mean_coefficient_mse  median_coefficient_mse  maximum_coefficient_mse  mse_above_1  mean_attempts  maximum_attempts  total_restarts  mean_evaluated_states  mean_fit_ms  median_fit_ms
Single-start descent + abort at 200         50                    49              0.028986                0.000034                 0.702086            0           1.96                11              48                1220.36    28.472600      25.461063
                 Horvath L-infinity         50                    50              0.000007                0.000002                 0.000096            0           1.00                 1               0                    NaN     5.962016      

In [3]:
image=plot_and_report()
print(json.dumps(json.loads((DATA_DIR/"validation.json").read_text()),indent=2))


{
  "processes": 50,
  "estimates_checked": 100,
  "actual_descent_attempts_checked": 98,
  "no_extra_draw_without_abort": true,
  "first_nonaborted_result_returned": true,
  "stationarity_roots_independently_checked": true,
  "all_50_lps_independently_solved_on_full_data": true,
  "maximum_lp_objective_difference": 2.0816681711721685e-13,
  "maximum_selected_width_error": 0.0,
  "all_timing_medians_checked": true,
  "timing_samples_checked": 500,
  "no_processes_dropped": true
}


Horváth has lower coefficient MSE on **44/50** paths; your descent on **6/50**. Aggregate MSE is approximately **7.02e-6** for Horváth versus **0.02899** for descent. Returned estimates are stationary in 50/50 and 49/50 cases respectively.

Your method makes **98 attempts** in total: 50 initial directions and 48 triggered resamples. Only 14 processes require resampling; 36 finish from their first direction. Average attempts are **1.96**, maximum 11. These are actual conditional draws, with no 12-start loop.

Measured mean fit times are approximately **5.96 ms** for Horváth and **28.47 ms** for descent, including preparation and discarded attempts. These describe the tested local implementations, rather than intrinsic complexity or a guarantee on other data. Descent minimizes normalized directional width while the benchmark minimizes maximum absolute residual; coefficient MSE supplies the common comparison target.

![Paired coefficient error and measured runtime](outputs/images/ar3_single_start_restart_vs_horvath/comparison.png)

[Full findings](outputs/csv/ar3_single_start_restart_vs_horvath/findings.md) · [All 100 estimates](outputs/csv/ar3_single_start_restart_vs_horvath/per_process_results.csv) · [Actual descent attempts](outputs/csv/ar3_single_start_restart_vs_horvath/descent_attempts.csv) · [Raw timing measurements](outputs/csv/ar3_single_start_restart_vs_horvath/timing_measurements.csv) · [Independent validation](outputs/csv/ar3_single_start_restart_vs_horvath/validation.json) · [PDF](outputs/pdfs/ar3_single_start_restart_vs_horvath/comparison.pdf)
